## Cell 1: Environment Setup & Configuration

In [ ]:
import os
import requests
import pandas as pd
from typing import List, Dict, Optional

FRED_API_KEY = os.getenv("FRED_API_KEY", "your_fred_api_key_here")  

# FRED Target Series
FRED_SERIES = [
    "FEDFUNDS",  
    "CPIAUCSL",  
    "UNRATE",    
    "GDPC1",     
    "GS10",      
    "M2SL"       
]

# World Bank Target Indicators
WB_INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_annual_pct",
    "FP.CPI.TOTL.ZG": "inflation_annual_pct",
    "FS.AST.PRVT.GD.ZS": "domestic_credit_to_private_pct_gdp",
    "FB.AST.NPL.ZS": "bank_npl_pct",
    "BX.TRF.PWKR.DT.GD.ZS": "remittances_received_pct_gdp"
}

# Target countries (ISO3 format) and timeframe
TARGET_COUNTRIES = ["USA", "EGY", "SAU", "ARE", "GBR"]
START_YEAR = 2000
END_YEAR = 2024

## Cell 2: FRED Ingestion Function

In [13]:
def fetch_fred_series(
    series_list: List[str], 
    api_key: str, 
    observation_start: str = "2000-01-01"
) -> pd.DataFrame:
    """
    Fetches historical time series observations from the FRED API.
    
    Optimizations:
    - Uses requests.Session for TCP connection pooling.
    - Handles missing/null string markers ('.') gracefully.
    """
    base_url = "https://api.stlouisfed.org/fred/series/observations"
    records: List[Dict] = []
    
    with requests.Session() as session:
        for series_id in series_list:
            params = {
                "series_id": series_id,
                "api_key": api_key,
                "file_type": "json",
                "observation_start": observation_start
            }
            try:
                response = session.get(base_url, params=params, timeout=15)
                response.raise_for_status()
                payload = response.json()
                
                observations = payload.get("observations", [])
                for obs in observations:
                    raw_val = obs.get("value")
                    # Handle FRED's default missing value representation ('.')
                    numeric_val = float(raw_val) if raw_val and raw_val != "." else None
                    
                    records.append({
                        "series_id": series_id,
                        "date": obs.get("date"),
                        "value": numeric_val
                    })
            except requests.exceptions.RequestException as err:
                print(f"[ERROR] Failed to retrieve FRED series '{series_id}': {err}")

    df = pd.DataFrame(records)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"])
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    
    return df

## Cell 3: World Bank Ingestion Function

In [14]:
import time

def fetch_world_bank_indicators(
    indicators: Dict[str, str], 
    countries: List[str], 
    start_year: int, 
    end_year: int
) -> pd.DataFrame:
    """
    Fetches country-level indicator data from the World Bank API v2.
    
    Optimizations:
    - Semicolon-delimited country batches.
    - Extended timeout and retry mechanism for resilient network requests.
    """
    countries_param = ";".join(countries)
    records: List[Dict] = []
    
    with requests.Session() as session:
        for ind_code, ind_name in indicators.items():
            url = f"http://api.worldbank.org/v2/country/{countries_param}/indicator/{ind_code}"
            params = {
                "date": f"{start_year}:{end_year}",
                "format": "json",
                "per_page": 1000
            }
            
            # Retry up to 3 times in case of World Bank server latency
            success = False
            for attempt in range(1, 4):
                try:
                    response = session.get(url, params=params, timeout=30)
                    response.raise_for_status()
                    payload = response.json()
                    
                    if len(payload) > 1 and payload[1]:
                        for item in payload[1]:
                            records.append({
                                "country_iso3": item.get("countryiso3code"),
                                "country_name": item["country"]["value"] if item.get("country") else None,
                                "indicator_code": ind_code,
                                "indicator_name": ind_name,
                                "year": int(item["date"]),
                                "value": float(item["value"]) if item.get("value") is not None else None
                            })
                    success = True
                    break
                except (requests.exceptions.RequestException, requests.exceptions.Timeout) as err:
                    if attempt < 3:
                        time.sleep(2)
                    else:
                        print(f"[ERROR] Failed to retrieve World Bank indicator '{ind_code}' after 3 attempts: {err}")

    df = pd.DataFrame(records)
    if not df.empty:
        df["year"] = df["year"].astype(int)
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        
    return df

## Cell 4: Execute Ingestion & Validate Dataframes

In [ ]:
# Execute FRED ingestion
print("Executing FRED API extraction...")
df_fred = fetch_fred_series(
    series_list = FRED_SERIES, 
    api_key = "FRED_API_KEY", 
    observation_start = f"{START_YEAR}-01-01"
)
print(f"-> FRED extracted: {len(df_fred):,} records.")

# Execute World Bank ingestion
print("\nExecuting World Bank API extraction...")
df_wb = fetch_world_bank_indicators(
    indicators=WB_INDICATORS, 
    countries=TARGET_COUNTRIES, 
    start_year=START_YEAR, 
    end_year=END_YEAR
)
print(f"-> World Bank extracted: {len(df_wb):,} records.")

Executing FRED API extraction...
-> FRED extracted: 1,705 records.

Executing World Bank API extraction...
-> World Bank extracted: 500 records.


## Cell 5: Quick Inspection & Profiling

In [16]:
# Display sample records and basic structure for FRED
print("=== FRED Sample ===")
display(df_fred.head())
print("\n=== FRED Data Types & Null Counts ===")
print(df_fred.info())

# Display sample records and basic structure for World Bank
print("\n=== World Bank Sample ===")
display(df_wb.head())
print("\n=== World Bank Data Types & Null Counts ===")
print(df_wb.info())

=== FRED Sample ===


,series_id,date,value
0,FEDFUNDS,2000-01-01,5.45
1,FEDFUNDS,2000-02-01,5.73
2,FEDFUNDS,2000-03-01,5.85
3,FEDFUNDS,2000-04-01,6.02
4,FEDFUNDS,2000-05-01,6.27



=== FRED Data Types & Null Counts ===
<class 'pandas.DataFrame'>
RangeIndex: 1705 entries, 0 to 1704
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   series_id  1705 non-null   str           
 1   date       1705 non-null   datetime64[us]
 2   value      1703 non-null   float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 40.1 KB
None

=== World Bank Sample ===


,country_iso3,country_name,indicator_code,indicator_name,year,value
0,ARE,United Arab Emirates,NY.GDP.MKTP.KD.ZG,gdp_growth_annual_pct,2024,3.991812
1,ARE,United Arab Emirates,NY.GDP.MKTP.KD.ZG,gdp_growth_annual_pct,2023,4.301136
2,ARE,United Arab Emirates,NY.GDP.MKTP.KD.ZG,gdp_growth_annual_pct,2022,7.514524
3,ARE,United Arab Emirates,NY.GDP.MKTP.KD.ZG,gdp_growth_annual_pct,2021,4.552801
4,ARE,United Arab Emirates,NY.GDP.MKTP.KD.ZG,gdp_growth_annual_pct,2020,-8.693434



=== World Bank Data Types & Null Counts ===
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country_iso3    500 non-null    str    
 1   country_name    500 non-null    str    
 2   indicator_code  500 non-null    str    
 3   indicator_name  500 non-null    str    
 4   year            500 non-null    int64  
 5   value           439 non-null    float64
dtypes: float64(1), int64(1), str(4)
memory usage: 23.6 KB
None


## Cell 6: Cell 6: PostgreSQL Connection, Raw Schema DDL & Data Ingestion

In [ ]:
from sqlalchemy import create_engine, text

# 1. Database Connection Settings
DB_USER = "postgres"
DB_PASS = "YOUR_PASSWORD_HERE"  
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "economic_bi_db"

connection_uri = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_uri)

# 2. Create raw schema and staging tables DDL
ddl_script = """
CREATE SCHEMA IF NOT EXISTS raw;

CREATE TABLE IF NOT EXISTS raw.fred_observations (
    series_id VARCHAR(50) NOT NULL,
    observation_date DATE NOT NULL,
    value NUMERIC,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (series_id, observation_date)
);

CREATE TABLE IF NOT EXISTS raw.world_bank_indicators (
    country_iso3 VARCHAR(3) NOT NULL,
    country_name VARCHAR(100),
    indicator_code VARCHAR(100) NOT NULL,
    indicator_name VARCHAR(100),
    observation_year INT NOT NULL,
    value NUMERIC,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (country_iso3, indicator_code, observation_year)
);
"""

# Execute table schema creation
with engine.connect() as conn:
    conn.execute(text(ddl_script))
    conn.commit()
    print("[SUCCESS] Schema 'raw' and staging tables created successfully.")

# 3. Load DataFrames into PostgreSQL Staging Tables
print("\nLoading FRED dataset to raw.fred_observations...")
df_fred_clean = df_fred.rename(columns={"date": "observation_date"})
df_fred_clean.to_sql(
    name="fred_observations",
    schema="raw",
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000
)
print(f"-> FRED loaded successfully: {len(df_fred_clean):,} rows.")

print("\nLoading World Bank dataset to raw.world_bank_indicators...")
df_wb_clean = df_wb.rename(columns={"year": "observation_year"})
df_wb_clean.to_sql(
    name="world_bank_indicators",
    schema="raw",
    con=engine,
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=1000
)
print(f"-> World Bank loaded successfully: {len(df_wb_clean):,} rows.")

[SUCCESS] Schema 'raw' and staging tables created successfully.

Loading FRED dataset to raw.fred_observations...
-> FRED loaded successfully: 1,705 rows.

Loading World Bank dataset to raw.world_bank_indicators...
-> World Bank loaded successfully: 500 rows.


## Cell 7: Database Verification & Row Count Integrity Check

In [18]:
with engine.connect() as conn:
    fred_db_count = conn.execute(text("SELECT COUNT(*) FROM raw.fred_observations;")).scalar()
    wb_db_count = conn.execute(text("SELECT COUNT(*) FROM raw.world_bank_indicators;")).scalar()

print("=== PostgreSQL Staging Row Count Verification ===")
print(f"raw.fred_observations   : {fred_db_count:,} rows (Expected: 1,705)")
print(f"raw.world_bank_indicators: {wb_db_count:,} rows (Expected: 500)")

=== PostgreSQL Staging Row Count Verification ===
raw.fred_observations   : 1,705 rows (Expected: 1,705)
raw.world_bank_indicators: 500 rows (Expected: 500)
